# Vegan Shop - Command-Line Manager

Final project for the **Python Programming** module. This notebook implements a small command-line management tool for a vegan shop, with persistent inventory, sale registration, and profit tracking.

This version is ready for GitHub publication: the code is organized into functions and classes, the cells are reproducible, the interactive loop does not start automatically, and data files are loaded robustly.

## Features

- Add new products or increase existing stock.
- Record one or more product sales and update inventory automatically.
- Calculate cumulative gross revenue and net profit.
- Persist data in `shop_storage.json` and `shop_profits.txt`.
- Load both the original project data format and a clearer key-based format.
- Run a reproducible demo and automated checks without manual input.

## Setup

The project uses only the Python standard library. To run the notebook, you only need Python 3.10+ and Jupyter. The files `shop_storage.json` and `shop_profits.txt` must be in the same folder as the notebook.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import json

In [2]:
INVENTORY_PATH = Path("shop_storage.json")
PROFITS_PATH = Path("shop_profits.txt")

## Data Model

The following classes represent products in inventory and the shop's financial summary.

In [3]:
def normalize_name(name: str) -> str:
    """Return a consistent product name for comparisons and storage."""
    clean_name = name.strip().lower()
    if not clean_name:
        raise ValueError("Product name cannot be empty.")
    return clean_name


def validate_positive_int(value: int, field_name: str) -> int:
    try:
        parsed_value = int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{field_name} must be an integer.") from exc

    if parsed_value <= 0:
        raise ValueError(f"{field_name} must be greater than zero.")
    return parsed_value


def validate_positive_float(value: float, field_name: str) -> float:
    try:
        parsed_value = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{field_name} must be a number.") from exc

    if parsed_value <= 0:
        raise ValueError(f"{field_name} must be greater than zero.")
    return parsed_value

In [4]:
@dataclass(slots=True)
class Product:
    quantity: int
    purchase_price: float
    selling_price: float

    def __post_init__(self) -> None:
        self.quantity = validate_positive_int(self.quantity, "quantity")
        self.purchase_price = validate_positive_float(self.purchase_price, "purchase price")
        self.selling_price = validate_positive_float(self.selling_price, "selling price")

    @property
    def margin(self) -> float:
        return self.selling_price - self.purchase_price

    def to_json(self) -> dict[str, float | int]:
        return {
            "quantity": self.quantity,
            "purchase_price": self.purchase_price,
            "selling_price": self.selling_price,
        }


@dataclass(slots=True)
class ProfitSummary:
    gross: float = 0.0
    net: float = 0.0

    def add_sale(self, product: Product, quantity: int) -> None:
        self.gross += product.selling_price * quantity
        self.net += product.margin * quantity

    def to_tuple(self) -> tuple[float, float]:
        return round(self.gross, 2), round(self.net, 2)

## Loading and Saving Data

The I/O functions support both the original `shop_storage.json` format and a more explicit format based on descriptive keys.

In [5]:
def product_from_record(product_name: str, record: object) -> Product:
    """Convert a JSON record into a Product object."""
    try:
        if isinstance(record, dict):
            return Product(
                quantity=record["quantity"],
                purchase_price=record["purchase_price"],
                selling_price=record["selling_price"],
            )

        quantity, prices = record
        purchase_price, selling_price = prices
        return Product(quantity, purchase_price, selling_price)
    except (KeyError, TypeError, ValueError) as exc:
        raise ValueError(f"Invalid record for {product_name!r}: {record!r}") from exc


def load_inventory(path: str | Path = INVENTORY_PATH) -> dict[str, Product]:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return {}

    raw_inventory = json.loads(path.read_text(encoding="utf-8"))
    return {
        normalize_name(product_name): product_from_record(product_name, record)
        for product_name, record in raw_inventory.items()
    }


def save_inventory(inventory: dict[str, Product], path: str | Path = INVENTORY_PATH) -> None:
    path = Path(path)
    serializable_inventory = {
        product_name: product.to_json()
        for product_name, product in sorted(inventory.items())
    }
    path.write_text(
        json.dumps(serializable_inventory, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def load_profits(path: str | Path = PROFITS_PATH) -> ProfitSummary:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return ProfitSummary()

    values = path.read_text(encoding="utf-8").split()
    if len(values) != 2:
        raise ValueError(f"The file {path} must contain two numbers: gross and net.")

    gross, net = map(float, values)
    return ProfitSummary(gross=gross, net=net)


def save_profits(profits: ProfitSummary, path: str | Path = PROFITS_PATH) -> None:
    gross, net = profits.to_tuple()
    Path(path).write_text(f"{gross:.2f} {net:.2f}", encoding="utf-8")

## Business Logic

The `VeganShop` class contains the main operations of the shop manager. Keeping the business logic separate from the `input()` loop makes the code easier to test and reuse.

In [6]:
class VeganShop:
    def __init__(
        self,
        inventory: dict[str, Product] | None = None,
        profits: ProfitSummary | None = None,
        inventory_path: str | Path = INVENTORY_PATH,
        profits_path: str | Path = PROFITS_PATH,
    ) -> None:
        self.inventory = dict(inventory or {})
        self.profits = profits or ProfitSummary()
        self.inventory_path = Path(inventory_path)
        self.profits_path = Path(profits_path)

    @classmethod
    def load(
        cls,
        inventory_path: str | Path = INVENTORY_PATH,
        profits_path: str | Path = PROFITS_PATH,
    ) -> "VeganShop":
        return cls(
            inventory=load_inventory(inventory_path),
            profits=load_profits(profits_path),
            inventory_path=inventory_path,
            profits_path=profits_path,
        )

    def save(self) -> None:
        save_inventory(self.inventory, self.inventory_path)
        save_profits(self.profits, self.profits_path)

    def add_product(
        self,
        name: str,
        quantity: int,
        purchase_price: float | None = None,
        selling_price: float | None = None,
    ) -> Product:
        product_name = normalize_name(name)
        quantity = validate_positive_int(quantity, "quantity")

        if product_name in self.inventory:
            product = self.inventory[product_name]
            product.quantity += quantity
            if purchase_price is not None:
                product.purchase_price = validate_positive_float(purchase_price, "purchase price")
            if selling_price is not None:
                product.selling_price = validate_positive_float(selling_price, "selling price")
            return product

        if purchase_price is None or selling_price is None:
            raise ValueError("A new product requires both purchase price and selling price.")

        product = Product(quantity, purchase_price, selling_price)
        self.inventory[product_name] = product
        return product

    def sell_product(self, name: str, quantity: int) -> dict[str, float | int | str]:
        product_name = normalize_name(name)
        quantity = validate_positive_int(quantity, "quantity")

        if product_name not in self.inventory:
            raise KeyError(f"{product_name!r} is not in inventory.")

        product = self.inventory[product_name]
        if quantity > product.quantity:
            raise ValueError("Requested quantity is greater than available stock.")

        receipt = {
            "product": product_name,
            "quantity": quantity,
            "unit_price": round(product.selling_price, 2),
            "total": round(product.selling_price * quantity, 2),
        }

        self.profits.add_sale(product, quantity)
        product.quantity -= quantity
        if product.quantity == 0:
            del self.inventory[product_name]

        return receipt

    def inventory_rows(self) -> list[dict[str, float | int | str]]:
        return [
            {
                "product": product_name,
                "quantity": product.quantity,
                "purchase_price": round(product.purchase_price, 2),
                "selling_price": round(product.selling_price, 2),
                "margin": round(product.margin, 2),
            }
            for product_name, product in sorted(self.inventory.items())
        ]

    def profit_report(self) -> dict[str, float]:
        gross, net = self.profits.to_tuple()
        return {"gross": gross, "net": net}

## Formatting and Command-Line Interface

The notebook does not start the terminal interface automatically, so it can run on GitHub or in Jupyter without getting stuck on `input()`. To use the shop manager locally, run `run_cli()` in the final dedicated cell.

In [7]:
def money(value: float) -> str:
    return f"EUR {value:.2f}"


def format_inventory(rows: list[dict[str, float | int | str]]) -> str:
    if not rows:
        return "Inventory is empty."

    header = f"{'Product':<16} {'Qty':>6} {'Purchase':>12} {'Sale':>12} {'Margin':>12}"
    separator = "-" * len(header)
    body = [
        f"{row['product']:<16} {row['quantity']:>6} {money(row['purchase_price']):>12} {money(row['selling_price']):>12} {money(row['margin']):>12}"
        for row in rows
    ]
    return "\n".join([header, separator, *body])


def format_receipt(receipt: dict[str, float | int | str]) -> str:
    return (
        f"- {receipt['quantity']} x {receipt['product']} "
        f"at {money(receipt['unit_price'])}: {money(receipt['total'])}"
    )


def format_profit_report(report: dict[str, float]) -> str:
    return f"Gross revenue: {money(report['gross'])} | Net profit: {money(report['net'])}"

In [8]:
HELP_TEXT = """
Available commands:
- add: add a new product or increase existing stock
- list: show available products
- sell: record a sale
- profits: show gross revenue and net profit
- help: show this message
- exit: save and close the program
""".strip()


def ask_positive_int(prompt: str) -> int:
    return validate_positive_int(input(prompt), "quantity")


def ask_positive_float(prompt: str, field_name: str) -> float:
    return validate_positive_float(input(prompt), field_name)


def run_cli(
    inventory_path: str | Path = INVENTORY_PATH,
    profits_path: str | Path = PROFITS_PATH,
) -> None:
    shop = VeganShop.load(inventory_path, profits_path)
    print(HELP_TEXT)

    while True:
        command = input("\nEnter a command: " ).strip().lower()

        try:
            if command == "add":
                name = input("Product name: " )
                quantity = ask_positive_int("Quantity: " )

                if normalize_name(name) in shop.inventory:
                    shop.add_product(name, quantity)
                else:
                    purchase_price = ask_positive_float("Purchase price: " , "purchase price")
                    selling_price = ask_positive_float("Selling price: " , "selling price")
                    shop.add_product(name, quantity, purchase_price, selling_price)

                print(f"ADDED: {quantity} x {normalize_name(name)}")

            elif command == "list":
                print(format_inventory(shop.inventory_rows()))

            elif command == "sell":
                receipt_lines = []
                while True:
                    name = input("Product name: " )
                    quantity = ask_positive_int("Quantity: " )
                    receipt_lines.append(format_receipt(shop.sell_product(name, quantity)))

                    another = input("Add another product? [yes/NO] " ).strip().lower()
                    if another != "yes":
                        break

                print("SALE RECORDED")
                print("\n".join(receipt_lines))

            elif command == "profits":
                print(format_profit_report(shop.profit_report()))

            elif command == "help":
                print(HELP_TEXT)

            elif command == "exit":
                shop.save()
                print("Data saved. Goodbye!")
                break

            else:
                print("Invalid command. Type 'help' to see the available options.")

        except (KeyError, ValueError) as exc:
            print(f"Error: {exc}")

## Current Shop State

This cell reads the data files in the project folder and shows the current inventory without modifying it.

In [9]:
shop = VeganShop.load()
print(format_inventory(shop.inventory_rows()))
print(format_profit_report(shop.profit_report()))

Product             Qty     Purchase         Sale       Margin
--------------------------------------------------------------
beans                26     EUR 0.30     EUR 0.80     EUR 0.50
tofu                  2     EUR 1.40     EUR 3.00     EUR 1.60
Gross revenue: EUR 8.00 | Net profit: EUR 5.00


## Reproducible Demo

The following demo uses in-memory data, so it can be run repeatedly without changing `shop_storage.json` or `shop_profits.txt`.

In [10]:
demo_shop = VeganShop(
    inventory={
        "tofu": Product(quantity=2, purchase_price=1.4, selling_price=3.0),
        "beans": Product(quantity=26, purchase_price=0.3, selling_price=0.8),
    },
    profits=ProfitSummary(gross=8.0, net=5.0),
)

demo_shop.add_product("seitan", quantity=3, purchase_price=2.5, selling_price=4.5)
receipt = demo_shop.sell_product("beans", quantity=4)

print("Sample sale")
print(format_receipt(receipt))
print()
print(format_inventory(demo_shop.inventory_rows()))
print(format_profit_report(demo_shop.profit_report()))

Sample sale
- 4 x beans at EUR 0.80: EUR 3.20

Product             Qty     Purchase         Sale       Margin
--------------------------------------------------------------
beans                22     EUR 0.30     EUR 0.80     EUR 0.50
seitan                3     EUR 2.50     EUR 4.50     EUR 2.00
tofu                  2     EUR 1.40     EUR 3.00     EUR 1.60
Gross revenue: EUR 11.20 | Net profit: EUR 7.00


## Automated Checks

These checks verify the core rules: adding a product, recording a sale, updating profits, and blocking sales above available stock.

In [11]:
test_shop = VeganShop()
test_shop.add_product("tofu", quantity=10, purchase_price=1.4, selling_price=3.0)
receipt = test_shop.sell_product("tofu", quantity=3)

assert receipt == {"product": "tofu", "quantity": 3, "unit_price": 3.0, "total": 9.0}
assert test_shop.inventory["tofu"].quantity == 7
assert test_shop.profit_report() == {"gross": 9.0, "net": 4.8}

try:
    test_shop.sell_product("tofu", quantity=100)
except ValueError as exc:
    assert "greater" in str(exc)
else:
    raise AssertionError("Selling above available stock should have failed.")

print("All checks passed.")

All checks passed.


## Running the Shop Manager

To use the program interactively, uncomment the line in the cell below and run it locally. When you close the program with the `exit` command, the data is saved to disk.

In [12]:
# run_cli()